# segmentation_prep.ipynb

**Langkah 1 dari pipeline: standardisasi dataset.**

Dataset People Segmentation (Supervisely) memiliki 5.110 gambar training dan 568 gambar validasi (total 5.678 pasangan gambar/mask). `process_dataset` menggunakan **semua** gambar tanpa seleksi, mengganti namanya menjadi `photo-1 ... photo-N`, dan memperbarui file teks pembagian.

> **Notebook ini bersifat destruktif.** Notebook ini mengganti nama file dan menulis ulang dataset di tempat. Simpan salinan terpisah dari unduhan asli jika Anda mungkin ingin memulai ulang nantinya. Sel eksekusi dijaga oleh `RUN_PREP = False`.

In [1]:
import os
import cv2
from glob import glob
from shutil import move

In [2]:
def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

### Visualisasi dataset

Dua figur: distribusi ukuran file seluruh dataset (`01_file_size_distribution.png`) dan grid sampel gambar (`02_selected_samples.png`).

In [3]:
def visualize_dataset(images_dir, all_sizes, save_dir="plots"):
    """
    Figur dua panel:
        Kiri  : distribusi ukuran file di semua gambar
        Kanan : grid 3x3 sampel gambar dari dataset
    Disimpan ke plots/01_file_size_distribution.png dan plots/02_selected_samples.png.
    """
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import numpy as np

    create_dir(save_dir)
    num_total = len(all_sizes)

    sizes_kb = [s / 1024 for _, s in all_sizes]

    # -- Plot 1: Distribusi ukuran file
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.bar(range(len(sizes_kb)), sizes_kb, color="#2196F3", width=1.0, edgecolor="none")
    ax.set_xlabel("Peringkat gambar (diurutkan berdasarkan ukuran file, terbesar dahulu)", fontsize=11)
    ax.set_ylabel("Ukuran file (KB)", fontsize=11)
    ax.set_title(
        f"Dataset Lengkap: Distribusi Ukuran File\n"
        f"Total {num_total:,} gambar (semua digunakan, tanpa seleksi)",
        fontsize=13, fontweight="bold"
    )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()

    out1 = os.path.join(save_dir, "01_file_size_distribution.png")
    plt.savefig(out1, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[Plot] Distribusi ukuran file disimpan: {out1}")

    # -- Plot 2: Sampel gambar
    selected_paths = sorted(glob(os.path.join(images_dir, "photo-*.jpg")))[:9]
    if len(selected_paths) < 1:
        return

    n_show = min(9, len(selected_paths))
    ncols  = min(3, n_show)
    nrows  = int(np.ceil(n_show / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.8, nrows * 3.5))
    axes = np.array(axes).reshape(-1) if n_show > 1 else [axes]

    for ax, img_path in zip(axes, selected_paths):
        img = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if img is None:
            ax.axis("off")
            continue
        img_rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        size_kb  = os.path.getsize(img_path) / 1024
        basename = os.path.basename(img_path)
        ax.imshow(img_rgb)
        ax.set_title(f"{basename}\n{size_kb:.1f} KB", fontsize=9, fontweight="bold")
        ax.axis("off")

    # Sembunyikan axes yang tidak terpakai
    for ax in axes[n_show:]:
        ax.axis("off")

    fig.suptitle(
        f"Sampel Gambar dari Dataset Lengkap ({num_total:,} gambar)\n"
        "(semua gambar digunakan tanpa seleksi)",
        fontsize=12, fontweight="bold", y=1.02
    )
    plt.tight_layout()

    out2 = os.path.join(save_dir, "02_selected_samples.png")
    plt.savefig(out2, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[Plot] Sampel gambar disimpan: {out2}")

### Prosedur standardisasi

Menggunakan **semua** gambar dalam dataset (5.110 train + 568 val). Tidak ada seleksi berdasarkan ukuran file.
Mengembalikan `all_sizes_sorted` untuk visualisasi di atas.

In [4]:
def process_dataset(root_dir):
    """
    1. Enumerasi semua gambar JPEG di root_dir/images/.
    2. Gunakan SEMUA gambar (tanpa seleksi/filter).
    3. Ganti nama gambar dan mask-nya menjadi photo-1 ... photo-N.
    4. Perbarui file teks pembagian segmentasi (train.txt, val.txt, trainval.txt).

    Mengembalikan all_sizes_sorted untuk visualisasi downstream.
    """
    images_dir = os.path.join(root_dir, "images")
    masks_dir  = os.path.join(root_dir, "masks")
    seg_dir    = os.path.join(root_dir, "segmentation")

    for d in (images_dir, masks_dir, seg_dir):
        if not os.path.isdir(d):
            raise FileNotFoundError(f"Direktori tidak ditemukan: {d}")

    image_files = [f for f in os.listdir(images_dir) if f.lower().endswith(".jpg")]
    num_images = len(image_files)
    if num_images == 0:
        raise ValueError("Tidak ada gambar JPEG yang ditemukan di direktori images.")

    print(f"[INFO] Ditemukan {num_images:,} gambar. Menggunakan semua tanpa seleksi.")

    # Urutkan berdasarkan ukuran file menurun (hanya untuk penamaan konsisten dan visualisasi)
    all_sizes = sorted(
        [(f, os.path.getsize(os.path.join(images_dir, f))) for f in image_files],
        key=lambda x: x[1],
        reverse=True,
    )
    all_bases = [os.path.splitext(f)[0] for f, _ in all_sizes]

    # Bangun pemetaan nama-lama -> nama-baru (semua gambar)
    mapping = {orig: f"photo-{i + 1}" for i, orig in enumerate(all_bases)}

    # Perbarui file teks pembagian segmentasi
    for split in ("train", "val", "trainval"):
        txt_path = os.path.join(seg_dir, f"{split}.txt")
        if not os.path.isfile(txt_path):
            continue
        with open(txt_path, "r") as fh:
            lines = [os.path.splitext(line.strip())[0] for line in fh if line.strip()]
        new_lines = [mapping[name] for name in lines if name in mapping]
        with open(txt_path, "w") as fh:
            fh.write("\n".join(new_lines))

    # Ganti nama gambar dan mask (semua gambar)
    # Gunakan temporary names untuk menghindari konflik nama
    temp_mapping = {}
    for orig, new in mapping.items():
        temp_name = f"__temp__{new}"
        src_img  = os.path.join(images_dir, orig + ".jpg")
        tmp_img  = os.path.join(images_dir, temp_name + ".jpg")
        src_mask = os.path.join(masks_dir,  orig + ".png")
        tmp_mask = os.path.join(masks_dir,  temp_name + ".png")
        if os.path.isfile(src_img):
            move(src_img, tmp_img)
        if os.path.isfile(src_mask):
            move(src_mask, tmp_mask)
        temp_mapping[temp_name] = new

    # Rename dari temporary ke final
    for temp_name, new in temp_mapping.items():
        tmp_img = os.path.join(images_dir, temp_name + ".jpg")
        dst_img = os.path.join(images_dir, new + ".jpg")
        tmp_mask = os.path.join(masks_dir, temp_name + ".png")
        dst_mask = os.path.join(masks_dir, new + ".png")
        if os.path.isfile(tmp_img):
            move(tmp_img, dst_img)
        if os.path.isfile(tmp_mask):
            move(tmp_mask, dst_mask)

    print(f"[PERSIAPAN] Menggunakan semua {num_images:,} gambar. Diganti nama menjadi photo-1 ... photo-{num_images}.")
    return all_sizes

### Eksekusi

Setel `RUN_PREP = True` untuk menjalankan standardisasi. Ini dijalankan sekali saja pada dataset dan **mengganti nama** semua file.

In [5]:
create_dir("plots")

DATASET_ROOT = "people_segmentation"

RUN_PREP = False  # setel True SEKALI untuk mengganti nama semua gambar (sudah dijalankan; biarkan False)

if RUN_PREP:
    all_sizes = process_dataset(DATASET_ROOT)
    num_total = len(all_sizes)
    visualize_dataset(os.path.join(DATASET_ROOT, "images"), all_sizes, save_dir="plots")
    try:
        from IPython.display import Image, display
        display(Image("plots/01_file_size_distribution.png"))
        display(Image("plots/02_selected_samples.png"))
    except Exception:
        pass
    print(f"Selesai. {num_total:,} pasangan siap di {DATASET_ROOT}/images|masks. Jalankan data.ipynb selanjutnya.")
else:
    print("Dilewati (setel RUN_PREP=True untuk menjalankan).")

Dilewati (setel RUN_PREP=True untuk menjalankan).
